In [15]:
import os
import numpy as np
from random import random

print(os.getcwd())  # Выведет текущую рабочую директорию

/Users/smetdenis/Work/smetdenis/ml-notes/grokking-dl


In [4]:
import pandas as pd
from srsly.msgpack import epoch

df = pd.read_csv('./IMDB_Dataset.csv')

In [78]:
from nltk.tokenize import TweetTokenizer


def get_tokens(text):
    tokenizer = TweetTokenizer()
    return tokenizer.tokenize(text.lower())


tokens = list(map(lambda x: set(get_tokens(x)), df.review))
len(tokens[0]), len(tokens[1]), len(tokens[2]), len(tokens)

(198, 119, 128, 50000)

In [90]:
tokens = list(map(lambda x: x.lower().split(" "), df.review))
len(tokens[0]), len(tokens[1]), len(tokens[2]), len(tokens), tokens[0][:10]

(307,
 162,
 166,
 50000,
 ['one',
  'of',
  'the',
  'other',
  'reviewers',
  'has',
  'mentioned',
  'that',
  'after',
  'watching'])

In [91]:
from collections import Counter
import math


def similar(target, num=3):
    target_index = word2index[target]
    scores = Counter()

    for word, index in word2index.items():
        raw_diff = weight_0_1[index] - (weight_0_1[target_index])
        square_diff = raw_diff ** 2
        scores[word] = - math.sqrt(sum(square_diff))

    return scores.most_common(num)

In [92]:
from collections import Counter

word_counter = Counter()

for sent in tokens:
    for word in sent:
        word_counter[word] -= 1

vocab = list(set(map(lambda x: x[0], word_counter.most_common())))
len(vocab)

392053

In [93]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

In [94]:
concatenated = list()
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""
    input_dataset.append(sent_indices)
concatenated = np.array(concatenated)

len(concatenated), len(input_dataset)

(11557297, 50000)

In [95]:
import random

random.shuffle(input_dataset)

In [96]:
np.random.seed(1)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


alpha = 0.05
epochs = 2

hidden_size = 50
window = 2
negative = 5

weight_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
weight_1_2 = np.zeros((len(vocab), hidden_size))

layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

for rev_i, review in enumerate(input_dataset * epochs):
    for target_i in range(len(review)):
        target_samples = ([review[target_i]]
                          + list(concatenated[(np.random.rand(negative) * len(concatenated)).astype('int').tolist()]))

        left_context = review[max(0, target_i - window):target_i]
        right_context = review[target_i + 1:min(len(review), target_i + window)]

        layer_1 = np.mean(weight_0_1[left_context + right_context], axis=0)
        layer_2 = sigmoid(layer_1.dot(weight_1_2[target_samples].T))

        layer_2_delta = layer_2 - layer_2_target
        layer_1_delta = layer_2_delta.dot(weight_1_2[target_samples])

        weight_0_1[left_context + right_context] -= layer_1_delta * alpha
        weight_1_2[target_samples] -= np.outer(layer_2_delta, layer_1) * alpha

    if (rev_i % 1000 == 0):
        print(
            "Progress:" + str(100 * rev_i / float(len(input_dataset) * epochs)) + "% | " +
            "Test: " + str(similar('terrible', 3))
        )


Progress:0.0% | Test: [('terrible', -0.0), ('amiable-seeming', -0.36888446071662584), ('submariner,', -0.3749471397984516)]
Progress:1.0% | Test: [('terrible', -0.0), ('horrible', -0.7944136465792618), ('child', -0.8777366101780347)]
Progress:2.0% | Test: [('terrible', -0.0), ('single', -1.2574747591893394), ('hero', -1.2673007669920284)]
Progress:3.0% | Test: [('terrible', -0.0), ('poor', -1.7503654558914767), ('fantastic', -1.7591992204203777)]
Progress:4.0% | Test: [('terrible', -0.0), ('terrific', -1.9381422728806412), ('unique', -1.9850400889138993)]
Progress:5.0% | Test: [('terrible', -0.0), ('terrific', -2.0735333710060657), ('fantastic', -2.111535420872682)]
Progress:6.0% | Test: [('terrible', -0.0), ('horrible', -2.2094381142400215), ('terrific', -2.31246840239409)]
Progress:7.0% | Test: [('terrible', -0.0), ('horrible', -2.2635448794980597), ('strange', -2.4921516894615743)]
Progress:8.0% | Test: [('terrible', -0.0), ('horrible', -2.268794513492528), ('silent', -2.58215629186

In [97]:
similar('terrible', 10), similar('love', 10), similar('beautiful', 10), similar('horrible', 10)

([('terrible', -0.0),
  ('horrible', -2.284877627718778),
  ('lousy', -3.273396851853226),
  ('dreadful', -3.418305053851695),
  ('brilliant', -3.7048318214444818),
  ('wonderful', -3.7485119257076898),
  ('horrendous', -3.9322621318101483),
  ('fantastic', -3.9439493993728094),
  ('pitiful', -3.9601776600641005),
  ('crappy', -3.985803321127087)],
 [('love', -0.0),
  ('sympathise', -5.410389695362377),
  ('adore', -5.629042780166002),
  ('interfere', -5.634448118421301),
  ('coping', -5.64647947907505),
  ('empathise', -5.651948912899812),
  ('fascination', -5.677113537580011),
  ('associate', -5.738705290922651),
  ('empathize', -5.788512146976967),
  ('despise', -5.791015498526842)],
 [('beautiful', -0.0),
  ('manipulative', -3.4404216821746565),
  ('gorgeous', -3.4986300721930266),
  ('sweet', -3.6180173828213307),
  ('creepy', -3.621954699567012),
  ('large,', -3.6448748190348437),
  ('meandering', -3.6553629940496455),
  ('stubborn', -3.65705677232281),
  ('ravishing', -3.6647643